# nAChR Variant Data — Merge, Standardize, Deduplicate & Analyze

## Sources
- `nachr_db_manual.xlsx` — 413 rows, primarily Human, 120 PMIDs
- `human_manual2.xlsx` — 458 rows, Human/Rat/Mouse, 81 PMIDs
- `mouse_data_manual.xlsx` — 309 rows, primarily Mouse, 48 PMIDs

**Goal:** Merge into one clean `final.xlsx`, standardize values, handle duplicates, compute stats.

## 1. Imports & Load Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths — notebook is in merging_data/, sources in parent dir
BASE = Path.cwd().parent if Path.cwd().name == 'merging_data' else Path.cwd()

human1 = pd.read_excel(BASE / 'nachr_db_manual.xlsx')
human2 = pd.read_excel(BASE / 'human_manual2.xlsx')
mouse  = pd.read_excel(BASE / 'mouse_data_manual.xlsx')

print(f'nachr_db_manual:  {human1.shape}')
print(f'human_manual2:    {human2.shape}')
print(f'mouse_data_manual: {mouse.shape}')

nachr_db_manual:  (413, 15)
human_manual2:    (458, 15)
mouse_data_manual: (309, 15)


## 2. Rename Columns & Drop Empty Column

In [2]:
for df, name in [(human1, 'human1'), (human2, 'human2'), (mouse, 'mouse')]:
    df.rename(columns={'Speices': 'Species'}, inplace=True)
    if 'Unnamed: 14' in df.columns:
        df.drop(columns=['Unnamed: 14'], inplace=True)
    df['source_file'] = name  # temporary provenance tag, removed before export

print('Columns:', list(human1.columns))

Columns: ['OID', 'Species', 'nAChR subunit', 'Modification type', 'AA position', 'Initial AA', 'New AA', 'Effect', 'Measuring Technique', 'Pathology', 'Reference(PMID)', 'Entry by', 'Correct?', 'DOI', 'source_file']


## 3. Standardization Maps

Edit these dictionaries to add new normalizations. They're applied to all three source files before merging.

In [3]:
# ===========================================================================
# SPECIES
# ===========================================================================
SPECIES_MAP = {
    'Human': 'Human', 'human': 'Human',
    'Mouse': 'Mouse', 'mouse': 'Mouse',
    'Rat': 'Rat', 'rat': 'Rat',
    '': np.nan, ' ': np.nan,
}

# ===========================================================================
# EFFECT
# ===========================================================================
EFFECT_MAP = {
    'GOF ': 'GOF', 'gof': 'GOF', 'GOF': 'GOF',
    'lof': 'LOF', 'LOF': 'LOF',
    'no net effect': 'No net effect',
    'No Net Effect': 'No net effect',
    'No net effect': 'No net effect',
    'LOF/GOF': 'LOF/GOF',
}

# ===========================================================================
# MODIFICATION TYPE
# ===========================================================================
MODTYPE_MAP = {
    'substitution': 'Substitution',
    'Substitution': 'Substitution',
    'Deletion': 'Deletion',
    'Insertion': 'Insertion',
    'Stop': 'Stop',
    'Frameshift': 'Frameshift',
}

# ===========================================================================
# MEASURING TECHNIQUE — component-level (applied after splitting on ';')
# ===========================================================================
TECHNIQUE_MAP = {
    # --- Patch-clamp family: unify hyphenation & capitalisation ---
    'Patch Clamp': 'Patch-clamp',
    'Patch-Clamp': 'Patch-clamp',
    'Patch clamp': 'Patch-clamp',
    'Patch-clamp recording': 'Patch-clamp',
    'Single Cell Patch Clamp': 'Patch-clamp',
    'HEK cell Patch-clamp': 'Patch-clamp',
    'Cell-attached Patch-clamp': 'Cell-attached patch-clamp',
    'Whole-cell Patch Clamp': 'Whole-cell patch clamp',
    'Whole-cell patch clamp': 'Whole-cell patch clamp',
    'Whole-cell voltage-clamp': 'Whole-cell patch clamp',
    'Whole-cell clamp': 'Whole-cell patch clamp',
    'Whole-cell': 'Whole-cell patch clamp',
    'Outside-out patch clamp': 'Outside-out patch clamp',
    # --- Single-channel family ---
    'Single Channel': 'Single-channel recording',
    'Single-Channel Recording': 'Single-channel recording',
    'Single-channel patch-clamp': 'Single-channel recording',
    'Single-channel patch-clamp recording': 'Single-channel recording',
    # --- Voltage clamp ---
    'Voltage clamp': 'Voltage-clamp',
    'Voltage-clamp': 'Voltage-clamp',
    # --- TEVC aliases ---
    'Two-electrode voltage clamp': 'TEVC',
    'Xenopus oocyte expression': 'TEVC',
    # --- Generic electrophysiology ---
    'Electrophysiology (unspecified)': 'Electrophysiology',
    # --- Binding assays ---
    'ACh Binding': 'ACh binding',
    '125I-alpha-bgt binding': 'Alpha-bgt binding',
    '125 alpha BTX binding': 'Alpha-bgt binding',
    '125 alpha BuTx binding': 'Alpha-bgt binding',
    'Cell surface binding': 'Alpha-bgt binding',
    # --- Other ---
    'RNS (3-Hz)': 'RNS',
    'Electromyography': 'EMG',
    'Repetitive Nerve Stimulation': 'RNS',
    'HEK 293 expression': 'HEK cell expression',
    'NCS': 'NCS',
}

# ===========================================================================
# PATHOLOGY — component-level (applied after splitting on ';')
# ===========================================================================
PATHOLOGY_MAP = {
    # --- Case fixes ---
    'Nicotine Dependence': 'Nicotine dependence',
    # --- CMS family ---
    'Congenital myasthenic syndrome': 'CMS',
    'Congenital Myasthenia': 'CMS',
    'Slow-channel congenital myasthenic syndrome (SCCMS)': 'SCCMS',
    'Slow-channel congenital myasthenic syndrome': 'SCCMS',
    'Slow-channel syndrome': 'SCCMS',
    'Slow Channel Syndrome': 'SCCMS',
    # --- ALS family ---
    'SALS': 'sALS',
    'Sporadic ALS': 'sALS',
    # --- Epilepsy family ---
    'Sleep-related hyperkinetic epilepsy': 'SHE',
    'Sleep-related Hyperkinetic Epilepsy (SHE)': 'SHE',
    'Nocturnal frontal lobe epilepsy': 'ADNFLE',
    'Insular Epilepsy': 'Insular epilepsy',
    # --- Separator fixes (some entries use ' / ' instead of '; ') ---
    'FADS / LMPS': 'FADS; LMPS',
}

print('Standardization maps loaded.')

Standardization maps loaded.


## 4. Apply Standardization

Strip whitespace, map values, normalize compound fields (technique & pathology can contain `;`-separated lists).

In [4]:
def standardize_compound(value, mapping):
    """Split on ';', map each component, deduplicate, rejoin."""
    if pd.isna(value) or str(value).strip() == '':
        return np.nan
    parts = [p.strip() for p in str(value).replace(' / ', ';').split(';')]
    mapped = [mapping.get(p, p) for p in parts if p]
    seen = set()
    unique = []
    for m in mapped:
        if m not in seen:
            seen.add(m)
            unique.append(m)
    return '; '.join(unique) if unique else np.nan


def clean_dataframe(df):
    """Apply all standardizations to a dataframe."""
    # AA position: handle BEFORE .str.strip() which would kill int values in object columns
    if 'AA position' in df.columns:
        def clean_aa_pos(x):
            if pd.isna(x):
                return np.nan
            if isinstance(x, (int, float)):
                return str(int(x))
            return str(x).strip()
        df['AA position'] = df['AA position'].apply(clean_aa_pos)
    
    # Strip whitespace from all string columns
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()
    
    # Map categorical columns
    df['Species'] = df['Species'].map(SPECIES_MAP).fillna(df['Species'])
    df['Effect'] = df['Effect'].map(EFFECT_MAP).fillna(df['Effect'])
    df['Modification type'] = df['Modification type'].map(MODTYPE_MAP).fillna(df['Modification type'])
    
    # Standardize compound fields
    df['Measuring Technique'] = df['Measuring Technique'].apply(
        lambda x: standardize_compound(x, TECHNIQUE_MAP))
    df['Pathology'] = df['Pathology'].apply(
        lambda x: standardize_compound(x, PATHOLOGY_MAP))
    
    # Data types
    df['OID'] = pd.to_numeric(df['OID'], errors='coerce')
    df['Reference(PMID)'] = pd.to_numeric(df['Reference(PMID)'], errors='coerce').astype('Int64')
    
    # AA position: convert purely numeric ones to int, keep ranges/delimiters as strings
    if 'AA position' in df.columns:
        numeric_mask = df['AA position'].notna() & df['AA position'].str.match(r'^-?\d+$')
        df.loc[numeric_mask, 'AA position'] = pd.to_numeric(
            df.loc[numeric_mask, 'AA position'], errors='coerce').astype('Int64')
    
    return df

# Apply to all three
human1 = clean_dataframe(human1)
human2 = clean_dataframe(human2)
mouse  = clean_dataframe(mouse)

print('Standardization complete.')
n1 = human1['AA position'].isna().sum()
print(f'AA position NaN - human1: {n1}, human2: {human2["AA position"].isna().sum()}, mouse: {mouse["AA position"].isna().sum()}')
print(f'AA position dtype: {human1["AA position"].dtype}')
# Show any non-numeric values preserved
non_num = human1[human1['AA position'].notna() & ~human1['AA position'].apply(lambda x: str(x).lstrip('-').isdigit())]
if len(non_num) > 0:
    print(f'Non-numeric AA positions preserved: {non_num["AA position"].tolist()}')

Standardization complete.
AA position NaN - human1: 0, human2: 0, mouse: 0
AA position dtype: object
Non-numeric AA positions preserved: ['469-502']


## 5. Merge All Three

In [5]:
assert list(human1.columns) == list(human2.columns) == list(mouse.columns), "Column mismatch!"

raw = pd.concat([human1, human2, mouse], ignore_index=True)
raw.insert(0, 'row_id', range(1, len(raw) + 1))

print(f'Merged: {raw.shape[0]} rows')

Merged: 1180 rows


## 6. Build Variant Keys (NaN-safe)

AA positions are NaN for many rows (deletions, insertions, frameshifts, and substitutions where position wasn't recorded). Using `.astype(str)` would turn all NaN into the string `"nan"`, causing unrelated variants to falsely group together. Instead we use `row_id` as a unique placeholder.

In [6]:
VARIANT_COLS = ['Species', 'nAChR subunit', 'Modification type', 'AA position', 'Initial AA', 'New AA']

def make_variant_key(row):
    parts = []
    for col in VARIANT_COLS:
        val = row[col]
        if pd.isna(val):
            parts.append(f'NaN_{row["row_id"]}')
        else:
            parts.append(str(val))
    return '|'.join(parts)

raw['variant_key'] = raw.apply(make_variant_key, axis=1)
raw['variant_pmid_key'] = raw['variant_key'] + '|' + raw['Reference(PMID)'].astype(str)

n_nan = raw['AA position'].isna().sum()
print(f'Rows with NaN AA position: {n_nan}/{len(raw)} ({n_nan/len(raw)*100:.1f}%)')
print(f'Unique variant keys: {raw["variant_key"].nunique()}')

Rows with NaN AA position: 0/1180 (0.0%)


Unique variant keys: 841


## 7. Duplicate Analysis

| Level | Key | Action |
|---|---|---|
| **L1** — Exact duplicate | All data columns match | Drop (keep richest row) |
| **L2** — Same variant + same PMID | variant_key + PMID | Merge rows, combine pathology/technique. If Effect conflicts → flag for review |
| **L3** — Same variant, different PMID | variant_key only | Keep both — independent evidence |

In [7]:
# L1
cmp_cols = [c for c in raw.columns if c not in ('row_id', 'OID', 'source_file', 'variant_key', 'variant_pmid_key')]
l1_count = raw.duplicated(subset=cmp_cols, keep=False).sum()
print(f'L1 exact duplicates: {l1_count} rows involved')

# L2
l2_dup_count = raw['variant_pmid_key'].duplicated().sum()
print(f'L2 same variant+PMID: {l2_dup_count} duplicate rows')

# L3
l3_mask = raw.duplicated(subset='variant_key', keep=False) & ~raw.duplicated(subset='variant_pmid_key', keep=False)
print(f'L3 same variant different PMID: {l3_mask.sum()} rows across {raw.loc[l3_mask, "variant_key"].nunique()} variants')

L1 exact duplicates: 199 rows involved
L2 same variant+PMID: 255 duplicate rows
L3 same variant different PMID: 122 rows across 63 variants


## 8. Detect Effect Conflicts

In [8]:
def find_conflicts(df):
    conflicts = []
    for key, group in df.groupby('variant_key'):
        if len(group) <= 1:
            continue
        effects = group['Effect'].dropna().unique()
        paths = group['Pathology'].dropna().unique()
        pmids = group['Reference(PMID)'].dropna().unique()
        effect_conflict = len(effects) > 1
        path_conflict = len(paths) > 1
        if effect_conflict or path_conflict:
            conflicts.append({
                'variant_key': key,
                'effects_seen': ' | '.join(str(e) for e in effects),
                'pathologies_seen': ' | '.join(str(p) for p in paths) if len(paths) else 'none',
                'n_pmids': len(pmids),
                'pmids': '; '.join(str(p) for p in pmids),
                'n_rows': len(group),
                'effect_conflict': effect_conflict,
                'path_conflict': path_conflict,
            })
    return pd.DataFrame(conflicts)

conflicts_df = find_conflicts(raw)
print(f'Variants with conflicts: {len(conflicts_df)}')
print(f'  Effect conflicts: {conflicts_df["effect_conflict"].sum()}')
print(f'  Pathology conflicts: {conflicts_df["path_conflict"].sum()}')

if conflicts_df['effect_conflict'].sum() > 0:
    print('\n=== Effect Conflicts (will be flagged for manual review) ===')
    display(conflicts_df[conflicts_df['effect_conflict']][['variant_key', 'effects_seen', 'pmids']])

Variants with conflicts: 60
  Effect conflicts: 34
  Pathology conflicts: 29

=== Effect Conflicts (will be flagged for manual review) ===


,variant_key,effects_seen,pmids
0,Human|CHRNA1|Substitution|272|P|A,GOF | LOF,18663134; 19279256; 19339970
2,Human|CHRNA2|Substitution|22|T|I,LOF | LOF/GOF,24467848
4,Human|CHRNA3|Substitution|385|R|H,GOF | LOF,19628475; 21486776
5,Human|CHRNA3|Substitution|388|S|F,GOF | LOF,19628475; 21486776
6,Human|CHRNA4|Substitution|182|W|A,LOF | No net effect,23742319; 24190916
7,Human|CHRNA4|Substitution|336|R|C,GOF | LOF,24385388; 26952864
8,Human|CHRNA4|Substitution|451|P|L,LOF | No net effect,24385388; 26952864
9,Human|CHRNA4|Substitution|487|R|Q,GOF | LOF,24385388; 26952864; 19628475
10,Human|CHRNA4|Substitution|88|W|A,LOF | No net effect,24637639; 24190916
11,Human|CHRNA5|Substitution|261|C|S,GOF | No net effect,32472188


## 9. Deduplicate

```
L1 exact duplicate? → Drop (keep richest row)
L2 same variant+PMID?
  ├─ Same effect? → Merge: combine pathologies & techniques
  └─ Different effect? → Flag EFFECT_CONFLICT, keep both for manual review
L3 same variant, different PMID? → Keep both (independent evidence, tagged in dedup_note)
```

In [9]:
def deduplicate(df):
    df = df.copy()
    df['review_flag'] = np.nan
    df['dedup_note'] = np.nan
    
    # Sort: most-complete rows first
    df['_completeness'] = df.notna().sum(axis=1)
    df = df.sort_values('_completeness', ascending=False)
    
    # ---- Step 1: Drop L1 exact duplicates ----
    cmp_cols = [c for c in df.columns if c not in (
        'row_id', 'OID', 'source_file', 'variant_key', 'variant_pmid_key',
        'review_flag', 'dedup_note', '_completeness')]
    before = len(df)
    df = df.drop_duplicates(subset=cmp_cols, keep='first')
    print(f'L1 dropped: {before - len(df)}')
    
    # ---- Step 2: Handle L2 (same variant + same PMID) ----
    df = df.sort_values(['variant_pmid_key', '_completeness'], ascending=[True, False])
    
    merged_rows = []
    for key, group in df.groupby('variant_pmid_key'):
        if len(group) == 1:
            merged_rows.append(group.iloc[0])
        else:
            effects = group['Effect'].dropna().unique()
            if len(effects) <= 1:
                # Safe merge — same effect, combine metadata from all sources
                best = group.iloc[0].copy()
                all_paths = group['Pathology'].dropna().unique()
                best['Pathology'] = '; '.join(all_paths) if len(all_paths) > 0 else np.nan
                all_techs = group['Measuring Technique'].dropna().unique()
                best['Measuring Technique'] = '; '.join(all_techs) if len(all_techs) > 0 else np.nan
                best['DOI'] = group['DOI'].dropna().iloc[0] if group['DOI'].notna().any() else np.nan
                sources = group['source_file'].unique()
                best['source_file'] = '+'.join(sources)
                best['dedup_note'] = f'merged {len(group)} rows, same variant+PMID'
                merged_rows.append(best)
            else:
                # Effect conflict → keep all, flag for manual review
                for _, row in group.iterrows():
                    row['review_flag'] = 'EFFECT_CONFLICT'
                    row['dedup_note'] = f'Effect conflict: {" vs ".join(str(e) for e in effects)}'
                    merged_rows.append(row)
    
    df = pd.DataFrame(merged_rows, columns=df.columns)
    print(f'Rows after L2 merge: {len(df)}')
    
    # ---- Step 3: Tag L3 (same variant, different PMID) — informational only ----
    for key, group in df.groupby('variant_key'):
        if len(group) > 1:
            pmids = group['Reference(PMID)'].dropna().unique()
            if len(pmids) > 1:
                for idx in group.index:
                    if pd.isna(df.at[idx, 'review_flag']):
                        df.at[idx, 'dedup_note'] = 'same variant in multiple papers'
    
    # ---- Cleanup ----
    df.drop(columns=['_completeness', 'variant_key', 'variant_pmid_key'], inplace=True, errors='ignore')
    return df

final = deduplicate(raw)
n_flagged = final['review_flag'].notna().sum()
print(f'\nFinal rows: {len(final)}')
print(f'EFFECT_CONFLICT flagged: {n_flagged}')
final.head(3)

L1 dropped: 101


Rows after L2 merge: 933

Final rows: 933
EFFECT_CONFLICT flagged: 16


,row_id,OID,Species,nAChR subunit,Modification type,AA position,Initial AA,New AA,Effect,Measuring Technique,Pathology,Reference(PMID),Entry by,Correct?,DOI,source_file,review_flag,dedup_note
809,810,NaN,Human,CHRNA10,Substitution,151,W,T,LOF,TEVC,NaN,41712615,HM,Yes,https://doi.org/10.1021/acs.jmedchem.5c03296,human2,NaN,NaN
810,811,NaN,Human,CHRNA10,Substitution,192,Y,T,LOF,TEVC,NaN,41712615,HM,Yes,https://doi.org/10.1021/acs.jmedchem.5c03296,human2,NaN,NaN
602,603,NaN,Human,CHRNA10,Substitution,7,L,H,No net effect,TEVC,NaN,26395518,HM,Yes,https://doi.org/10.1038/srep14261,human2,NaN,NaN


## 10. Save final.xlsx

Drop the `source_file` debug column. Two sheets: `all_variants` + `needs_review`.

In [10]:
OUTPUT_PATH = Path.cwd() / 'final.xlsx' if Path.cwd().name == 'merging_data' else BASE / 'merging_data' / 'final.xlsx'

# Remove source_file column from final output
final_out = final.drop(columns=['source_file'], errors='ignore')

# Reorder columns
col_order = [
    'row_id', 'Species', 'nAChR subunit', 'Modification type',
    'AA position', 'Initial AA', 'New AA', 'Effect',
    'Measuring Technique', 'Pathology', 'Reference(PMID)', 'DOI',
    'Entry by', 'Correct?', 'review_flag', 'dedup_note'
]
final_out = final_out[[c for c in col_order if c in final_out.columns]]

# Flagged rows for separate sheet
flagged = final_out[final_out['review_flag'].notna()]

# ---- Build unique_variants sheet (collapse multi-paper evidence) ----
V = ['Species', 'nAChR subunit', 'Modification type', 'AA position', 'Initial AA', 'New AA']

def collapse_group(group):
    """Collapse a group of rows (same variant) into one."""
    row = group.iloc[0].copy()
    row['Reference(PMID)'] = '; '.join(str(p) for p in group['Reference(PMID)'].dropna().unique())
    row['DOI'] = '; '.join(str(d) for d in group['DOI'].dropna().unique())
    all_paths = group['Pathology'].dropna().unique()
    row['Pathology'] = '; '.join(all_paths) if len(all_paths) > 0 else np.nan
    all_techs = group['Measuring Technique'].dropna().unique()
    row['Measuring Technique'] = '; '.join(all_techs) if len(all_techs) > 0 else np.nan
    row['n_papers'] = group['Reference(PMID)'].nunique()
    row['n_rows_merged'] = len(group)
    return row

unique_variants = final_out.groupby(V, dropna=False).apply(collapse_group).reset_index(drop=True)
unique_variants.drop(columns=['row_id', 'review_flag', 'dedup_note'], inplace=True, errors='ignore')

# Reorder: put n_papers after DOI
uv_cols = [c for c in unique_variants.columns if c not in ('n_papers', 'n_rows_merged')]
uv_cols = uv_cols[:12] + ['n_papers', 'n_rows_merged'] + uv_cols[12:]
unique_variants = unique_variants[[c for c in uv_cols if c in unique_variants.columns]]

print(f'Unique variants: {len(unique_variants)} (from {len(final_out)} total rows)')
print(f'Single-paper variants: {(unique_variants["n_papers"] == 1).sum()}')
print(f'Multi-paper variants:  {(unique_variants["n_papers"] > 1).sum()}')

# ---- Save all three sheets ----
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    final_out.to_excel(writer, sheet_name='all_variants', index=False)
    flagged.to_excel(writer, sheet_name='needs_review', index=False)
    unique_variants.to_excel(writer, sheet_name='unique_variants', index=False)

print(f'\nSaved: {OUTPUT_PATH}')
print(f'  all_variants:     {len(final_out)} rows (full provenance)')
print(f'  unique_variants:  {len(unique_variants)} rows (one per variant, n_papers column)')
print(f'  needs_review:     {len(flagged)} rows (effect conflicts)')

Unique variants: 841 (from 933 total rows)
Single-paper variants: 776
Multi-paper variants:  63



Saved: C:\Users\harik\Downloads\Computational Chemistry\Variant-Effect-Predictor-for-nAChRs\merging_data\final.xlsx
  all_variants:     933 rows (full provenance)
  unique_variants:  841 rows (one per variant, n_papers column)
  needs_review:     16 rows (effect conflicts)


---
## 11. Statistics & Summary

In [11]:
f = final_out
V = ['Species', 'nAChR subunit', 'Modification type', 'AA position', 'Initial AA', 'New AA']

print('=' * 55)
print('DATASET SUMMARY')
print('=' * 55)
print(f'Total rows:              {len(f)}')
print(f'Unique variants:         {f[V].drop_duplicates().shape[0]}')
print(f'Effect conflicts:        {f["review_flag"].notna().sum()}')
print(f'Unique PMIDs:            {f["Reference(PMID)"].nunique()}')
print(f'Unique DOIs:             {f["DOI"].nunique()}')
print(f'Pathology coverage:      {f["Pathology"].notna().sum()}/{len(f)} ({f["Pathology"].notna().sum()/len(f)*100:.1f}%)')

DATASET SUMMARY
Total rows:              933
Unique variants:         841
Effect conflicts:        16
Unique PMIDs:            209
Unique DOIs:             211
Pathology coverage:      309/933 (33.1%)


In [12]:
print('\n--- Species ---')
print(f['Species'].value_counts().to_string())
print(f'\n--- Species (unique variants) ---')
print(f.drop_duplicates(subset=V)['Species'].value_counts().to_string())


--- Species ---
Species
Human    669
Rat      175
Mouse     88
           1

--- Species (unique variants) ---
Species
Human    585
Rat      174
Mouse     81
           1


In [13]:
print('\n--- Effect ---')
print(f['Effect'].value_counts().to_string())
print(f'\n--- Effect by Species ---')
display(pd.crosstab(f['Species'], f['Effect']))
print(f'\n--- Effect by Subunit (top 10) ---')
top10 = f['nAChR subunit'].value_counts().head(10).index
display(pd.crosstab(f[f['nAChR subunit'].isin(top10)]['nAChR subunit'], 
                    f[f['nAChR subunit'].isin(top10)]['Effect']))


--- Effect ---
Effect
LOF              457
GOF              244
No net effect    214
LOF/GOF           18

--- Effect by Species ---


Effect,GOF,LOF,LOF/GOF,No net effect
Species,,,,
,0,1,0,0
Human,180,371,16,102
Mouse,31,30,2,25
Rat,33,55,0,87



--- Effect by Subunit (top 10) ---


Effect,GOF,LOF,LOF/GOF,No net effect
nAChR subunit,,,,
CHRNA1,45,90,0,1
CHRNA4,28,54,1,18
CHRNA6,18,8,1,23
CHRNA7,57,126,10,75
CHRNA9,1,11,0,15
CHRNB1,8,16,0,2
CHRNB2,19,20,3,14
CHRNB4,19,8,0,27
CHRND,5,29,2,16


In [14]:
print('\n--- Subunit ---')
print(f['nAChR subunit'].value_counts().to_string())
print(f'\n--- Subunit by Species ---')
display(pd.crosstab(f['Species'], f['nAChR subunit']))


--- Subunit ---
nAChR subunit
CHRNA7     268
CHRNA1     136
CHRNA4     101
CHRNE       88
CHRNB2      56
CHRNB4      54
CHRND       52
CHRNA6      50
CHRNA9      27
CHRNB1      26
CHRNA3      16
CHRNA10     13
CHRNB3      13
CHRNA5      12
CHRNG       11
CHRNA2      10

--- Subunit by Species ---


nAChR subunit,CHRNA1,CHRNA10,CHRNA2,CHRNA3,CHRNA4,CHRNA5,CHRNA6,CHRNA7,CHRNA9,CHRNB1,CHRNB2,CHRNB3,CHRNB4,CHRND,CHRNE,CHRNG
Species,,,,,,,,,,,,,,,,
,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
Human,110,3,10,11,59,11,39,185,9,25,40,8,36,36,82,5
Mouse,26,0,0,0,5,1,5,1,0,1,2,5,14,16,6,6
Rat,0,10,0,5,37,0,6,81,18,0,14,0,4,0,0,0


In [15]:
print('\n--- Measuring Technique (top 15) ---')
print(f['Measuring Technique'].value_counts().head(15).to_string())
print(f'\nUnique technique strings: {f["Measuring Technique"].nunique()}')


--- Measuring Technique (top 15) ---
Measuring Technique
TEVC                                                511
Patch-clamp                                         129
Single-channel recording                             88
Voltage-clamp; TEVC                                  35
RNS                                                  28
Voltage-clamp                                        24
Whole-cell patch clamp                               20
Patch-clamp; Single-channel recording                11
Electrophysiology                                    11
Single-channel recording; Patch-clamp                 9
TEVC; Single-channel recording                        6
Whole-cell patch clamp; Single-channel recording      6
Alpha-bgt binding                                     6
Single-channel kinetics                               5
DNA Sequencing                                        4

Unique technique strings: 40


In [16]:
print('\n--- Pathology (top 20) ---')
all_paths = f['Pathology'].dropna().str.split('; ').explode().str.strip()
print(all_paths.value_counts().head(20).to_string())


--- Pathology (top 20) ---
Pathology
CMS                                      91
Nicotine dependence                      76
SCCMS                                    40
ADNFLE                                   29
Nicotine addiction                       25
Schizophrenia                            14
sALS                                     12
Neuropathic Pain                          6
FCCMS                                     6
Cognitive/Neurodegenerative disorders     5
AChR Deficiency                           4
LMPS                                      4
FADS                                      4
Epilepsy                                  4
Mood disorders                            3
Escobar syndrome                          2
Neonatal lethality                        2
ADHD                                      2
Increased neuronal apoptosis              1
Hyperactivity                             1


In [17]:
print('\n--- Modification Type ---')
print(f['Modification type'].value_counts().to_string())
print(f'\n--- Mod Type by Effect ---')
display(pd.crosstab(f['Modification type'], f['Effect']))


--- Modification Type ---
Modification type
Substitution    903
Deletion         15
Stop             10
Frameshift        4
Insertion         1

--- Mod Type by Effect ---


Effect,GOF,LOF,LOF/GOF,No net effect
Modification type,,,,
Deletion,1,13,0,1
Frameshift,0,4,0,0
Insertion,1,0,0,0
Stop,0,10,0,0
Substitution,242,430,18,213


In [18]:
print('\n--- NaN AA Position by Modification Type ---')
nan_pos = f[f['AA position'].isna()]
if len(nan_pos) > 0:
    print(nan_pos['Modification type'].value_counts().to_string())
    print(f'\nTotal NaN AA positions: {len(nan_pos)} ({len(nan_pos)/len(f)*100:.1f}%)')
    print('(Valid: deletions, insertions, frameshifts, stops, and substitutions where position was not recorded)')
else:
    print('No NaN positions.')


--- NaN AA Position by Modification Type ---
No NaN positions.


In [19]:
print('\n--- Top 15 Papers by Variant Count ---')
for pmid, count in f['Reference(PMID)'].value_counts().head(15).items():
    doi = f.loc[f['Reference(PMID)'] == pmid, 'DOI'].dropna().iloc[0] if not f.loc[f['Reference(PMID)'] == pmid, 'DOI'].dropna().empty else 'N/A'
    print(f'  PMID {pmid}: {count} variants')


--- Top 15 Papers by Variant Count ---
  PMID 32364364: 37 variants
  PMID 20650284: 35 variants
  PMID 37239959: 28 variants
  PMID 25957813: 21 variants
  PMID 35600303: 19 variants
  PMID 29497086: 18 variants
  PMID 24886653: 16 variants
  PMID 19339660: 15 variants
  PMID 18398509: 15 variants
  PMID 19279256: 14 variants
  PMID 26340455: 14 variants
  PMID 8327511: 13 variants
  PMID 24478678: 13 variants
  PMID 25740413: 13 variants
  PMID 26340537: 13 variants


## 12. Sanity Checks

In [20]:
issues = []
for col in ['Species', 'Effect', 'nAChR subunit', 'Modification type']:
    n_null = f[col].isna().sum()
    if n_null > 0:
        issues.append(f'{col}: {n_null} NaN values')
    if f[col].dtype == object:
        empty = (f[col] == '').sum()
        if empty > 0:
            issues.append(f'{col}: {empty} empty strings')

# Check no residual unstandardized values
if 'substitution' in f['Modification type'].values:
    issues.append('Lowercase "substitution" still present')
if 'Nicotine Dependence' in f['Pathology'].dropna().values:
    issues.append('"Nicotine Dependence" (capital D) still present')

unknown_species = f[~f['Species'].isin(['Human', 'Mouse', 'Rat'])]
if len(unknown_species) > 0:
    if unknown_species['Species'].notna().any():
        issues.append(f'Unknown Species: {unknown_species["Species"].dropna().unique()}')

if issues:
    print('ISSUES:')
    for i in issues:
        print(f'  - {i}')
else:
    print('All sanity checks passed.')

ISSUES:
  - Species: 1 empty strings
  - Unknown Species: ['']


---
## 13. Notes

### Effect conflicts (`needs_review` sheet)
Same variant in the same paper reported with different effects. **Manual review needed** — check the original paper (DOI provided). Common causes:
- Different experimental conditions (e.g., different subunit background)
- Data entry error in one of the source files
- Context-dependent effect (may be genuinely mixed LOF/GOF)

### NaN AA positions
~35% of rows have no AA position. These are:
- Substitutions where position wasn't extracted from the paper
- Deletions, insertions, frameshifts, and stop codons (position isn't a single number)

The variant_key uses a row-level placeholder to avoid false grouping.

### When adding future data
- Add new rows to any of the three source files (or add a fourth source)
- Re-run this notebook — it re-merges and re-deduplicates from scratch
- Add any new technique/pathology variations to the maps in Section 3
- Always include PMID and DOI for provenance tracking

---
## DONE

`final.xlsx` saved with:
- **`all_variants`** — merged, standardized, deduplicated dataset
- **`needs_review`** — rows with conflicting effects for manual inspection